# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subikshasrig/FlyrankMLInternship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row represents the daily performance of a single content item (`content_hash_id`) for a specific client (`client_hash_id`) on one reporting date (`report_date`).

The analysis uses the March 2026 partition because it is a mid-panel month recommended for feature development. The final month (June 2026) is deliberately excluded to avoid developing on the natural outcome window.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
!pip -q install datasets huggingface_hub duckdb
from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import duckdb
print("Libraries loaded")
login(token=userdata.get("HF_TOKEN"))
print("Hugging Face login successful")

Libraries loaded
Hugging Face login successful


In [13]:
ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True
)
print(ds)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

IterableDataset({
    features: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'],
    num_shards: 18
})


In [14]:
sample = next(iter(ds))
print(sample.keys())

dict_keys(['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events'])


In [15]:
from huggingface_hub import hf_hub_download
march_file = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet"
)
print(march_file)

import duckdb
# Create a fresh connection
con = duckdb.connect()
# Register the dataframe as a SQL table
con.register("fact_content_daily_performance", march_df)
# Verify it exists
print(con.sql("SHOW TABLES").df())

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2/fact_content_daily_performance/month=2026-03/data_0.parquet
                             name
0  fact_content_daily_performance


In [16]:
con.sql("""
SELECT COUNT(*)
FROM fact_content_daily_performance
""").df()

,count_star()
0,9841378


In [17]:
con.sql("""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS row_count
FROM fact_content_daily_performance
GROUP BY
    report_date,
    client_hash_id,
    content_hash_id
HAVING COUNT(*) > 1
LIMIT 5
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,row_count


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Features
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

These are historical measurements available before a prediction is made.

### Label
The prediction target is future search performance (for example, future clicks or ranking improvement). Since future outcomes are not present in this monthly snapshot, they are not used as features.

### Context
- report_date
- client_hash_id
- content_hash_id
- month

These fields identify observations, support grouping, and define the analysis period but are not model features.

### Excluded
- client_has_gsc
- client_has_ga4
- gsc_data_available
- ga4_data_available

These fields describe data availability and system status rather than content performance. They are used for filtering and validation instead of prediction.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT *
FROM fact_content_daily_performance
LIMIT 5
""").df()

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## Verify it with queries (grain, counts, missing values, windows)

The data contract is verified using SQL queries rather than assumptions.

- Grain: I confirmed that each row represents a unique combination of `report_date`, `client_hash_id`, and `content_hash_id`. The duplicate check returned no rows, confirming the defined unit of analysis.
- Counts: The March 2026 partition contains 9,841,378 rows across **55** clients and **331,437** unique content items.
- Time window: The data spans 2026-03-01** to **2026-03-31, confirming that only the March 2026 partition is being analysed.
- Availability: I filtered the data using `ga4_data_available IS TRUE` to identify rows with valid GA4 metrics. This verifies that GA4 data is not available for every observation and should be filtered before using GA4-based features.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Grain, row count, and distinct entities
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items
FROM fact_content_daily_performance
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,clients,content_items
0,9841378,55,331437


In [20]:
# Date Window
con.sql("""
SELECT
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM fact_content_daily_performance
""").df()

,first_date,last_date
0,2026-03-01,2026-03-31


In [21]:
# Availability Check
con.sql("""
SELECT
    COUNT(*) AS available_rows
FROM fact_content_daily_performance
WHERE ga4_data_available IS TRUE
""").df()

,available_rows
0,413966


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This dataset has several limitations that should be considered when interpreting results.

- Client history is not balanced. Some clients have much longer data histories than others, which may affect comparisons across clients.
- GA4 metrics are only available when `ga4_data_available` is TRUE. Missing GA4 values do not necessarily mean zero activity.
- The warehouse contains historical observations only and cannot explain the causal reasons behind changes in search performance.
- This notebook uses only the March 2026 partition for development. The final month (June 2026) is intentionally excluded because it serves as the natural future evaluation period.

In [22]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
con.sql("""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
    COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable_rows
FROM fact_content_daily_performance
""").df()

,total_rows,ga4_available_rows,ga4_unavailable_rows
0,9841378,413966,9427412


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.